# Notebook 05: DuckDB — Feature Parity + Unique Syntax

**Phase 4 — DuckDB: Feature Parity + Unique Syntax**

DuckDB and PostgreSQL both speak SQL, but they are architecturally opposite:

| | PostgreSQL | DuckDB |
|:---|:---|:---|
| Storage | **Row-oriented** (heap) | **Columnar** |
| Execution | Row-at-a-time | Vectorized (SIMD, column batches) |
| Deployment | Server process, network socket | In-process, embedded |
| Target workload | OLTP — point lookups, writes, ACID | OLAP — aggregations, scans, analytics |
| Infrastructure | Docker / VM / managed cloud | `import duckdb` — zero setup |

**Why columnar is faster for aggregations:** `SUM(l_extendedprice)` on 6M
rows only touches the `l_extendedprice` column — all other columns are skipped
entirely.  Each column is homogeneous in type, so it compresses well (dictionary,
run-length, delta encoding) and the CPU can process 1024+ values per SIMD
instruction batch.  A row-oriented engine must read every column in every row
even when only one column is needed.

| Section | Topic |
|:---|:---|
| 1 | Re-running Phase 1 & 2 queries — identical SQL, DuckDB engine |
| 2 | DuckDB-exclusive syntax PostgreSQL cannot do |
| 3 | `postgres_scan` — query live PostgreSQL from DuckDB |

---

## Prerequisites

Notebook 01 must have been run first (Parquet files present).
PostgreSQL must be running (for Section 3).


In [ ]:
import pathlib
import sys

import duckdb
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import settings

PARQUET_DIR = ROOT / "data" / "parquet"

# In-memory DuckDB connection — no file, no server, no setup.
duck = duckdb.connect()

# Attach all TPC-H Parquet files as views so queries read exactly like the
# PostgreSQL versions — same table names, same column names.
tables = [
    "region", "nation", "supplier", "part",
    "partsupp", "customer", "orders", "lineitem",
]
for t in tables:
    duck.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{PARQUET_DIR / t}.parquet')"
    )

print(f"DuckDB {duckdb.__version__} ready — {len(tables)} TPC-H views attached")
print(f"Parquet dir: {PARQUET_DIR}")


---

## 1 · Re-running PostgreSQL Queries on DuckDB

The window function and CTE queries from Phases 1 & 2 run on DuckDB with
**zero SQL changes**.  Both engines implement the same ANSI SQL standard for
these constructs.

The point is not that one is faster here — it is that DuckDB is a fully
capable SQL engine that requires no infrastructure.  This has practical
consequences: local development, CI pipelines, one-off analysis, and unit
testing all work with `duckdb.connect()` rather than a running server.


In [ ]:
# Phase 1 query — ROW_NUMBER / RANK / DENSE_RANK on customer spend per nation.
# Identical SQL to 01_ranking.sql § ranking_window; engine is DuckDB not PostgreSQL.

ranking_sql = """
SELECT
    c.c_name,
    n.n_name                                              AS nation,
    SUM(o.o_totalprice)                                   AS total_spend,
    ROW_NUMBER() OVER w                                   AS row_num,
    RANK()       OVER w                                   AS rnk,
    DENSE_RANK() OVER w                                   AS dense_rnk
FROM   customer c
JOIN   orders   o  ON c.c_custkey   = o.o_custkey
JOIN   nation   n  ON c.c_nationkey = n.n_nationkey
GROUP  BY c.c_custkey, c.c_name, n.n_nationkey, n.n_name
WINDOW w AS (
    PARTITION BY n.n_nationkey
    ORDER BY SUM(o.o_totalprice) DESC
)
ORDER  BY nation, rnk
LIMIT  30
"""

duck.execute(ranking_sql).df().head(15)


In [ ]:
# Phase 2 query — chained CTEs, top nations by revenue (identical SQL to 01_chained_ctes.sql).
# DuckDB supports WITH ... AS MATERIALIZED from v0.9+.

chained_sql = """
WITH
high_priority_lines AS (
    SELECT
        l.l_orderkey,
        l.l_extendedprice * (1 - l.l_discount)  AS net_price
    FROM   lineitem l
    JOIN   orders   o ON l.l_orderkey = o.o_orderkey
    WHERE  o.o_orderpriority IN ('1-URGENT', '2-HIGH')
      AND  l.l_shipdate BETWEEN DATE '1995-01-01' AND DATE '1995-12-31'
),
order_revenue AS (
    SELECT
        o.o_custkey,
        SUM(hpl.net_price)  AS order_net_revenue
    FROM   high_priority_lines   hpl
    JOIN   orders                o   ON hpl.l_orderkey = o.o_orderkey
    GROUP  BY o.o_custkey
),
customer_nation AS (
    SELECT c.c_custkey, n.n_name AS nation, r.r_name AS region
    FROM   customer c
    JOIN   nation   n ON c.c_nationkey = n.n_nationkey
    JOIN   region   r ON n.n_regionkey = r.r_regionkey
),
nation_revenue AS (
    SELECT
        cn.region, cn.nation,
        SUM(orv.order_net_revenue)      AS total_net_revenue,
        COUNT(DISTINCT orv.o_custkey)   AS contributing_customers
    FROM   order_revenue   orv
    JOIN   customer_nation cn ON orv.o_custkey = cn.c_custkey
    GROUP  BY cn.region, cn.nation
)
SELECT
    nr.region,
    nr.nation,
    ROUND(nr.total_net_revenue, 2)  AS net_revenue_1995,
    nr.contributing_customers
FROM   nation_revenue nr
ORDER  BY net_revenue_1995 DESC
LIMIT  10
"""

duck.execute(chained_sql).df()


---

## 2 · DuckDB-Exclusive Syntax

These features exist in DuckDB but have no direct equivalent in PostgreSQL.


### `SELECT * EXCLUDE (col, ...)`

Drop specific columns without listing all others.  The standard workaround in
PostgreSQL is to list every column you want — brittle when the schema changes.


In [ ]:
# Without EXCLUDE: must list all 16 lineitem columns except l_comment
standard_sql = """
SELECT
    l_orderkey, l_partkey, l_suppkey, l_linenumber,
    l_quantity, l_extendedprice, l_discount, l_tax,
    l_returnflag, l_linestatus, l_shipdate, l_commitdate,
    l_receiptdate, l_shipinstruct, l_shipmode
FROM lineitem
LIMIT 3
"""

# With EXCLUDE: drop l_comment, keep everything else automatically
exclude_sql = "SELECT * EXCLUDE (l_comment) FROM lineitem LIMIT 3"

print("Standard (column list):")
print(duck.execute(standard_sql).df().to_string(index=False))
print()
print("EXCLUDE:")
print(duck.execute(exclude_sql).df().to_string(index=False))


### `SUMMARIZE`

Instant table profiling — min, max, mean, std, nulls, distinct count per
column in one expression.  The PostgreSQL equivalent requires writing a
multi-column aggregate query manually.


In [ ]:
# Profile the lineitem table — 16 columns, 6M rows, single expression.
duck.execute(f"SUMMARIZE SELECT * FROM read_parquet('{PARQUET_DIR}/lineitem.parquet')").df()


### Native `PIVOT`

PostgreSQL requires the `tablefunc` extension and the cumbersome `crosstab()`
function to pivot rows to columns.  DuckDB has a native `PIVOT` statement.


In [ ]:
# Pivot: order count per order status per year (1993-1998).
# PostgreSQL would need: CREATE EXTENSION tablefunc; SELECT * FROM crosstab(...)
pivot_sql = """
PIVOT (
    SELECT
        YEAR(o_orderdate)  AS order_year,
        o_orderstatus,
        COUNT(*)           AS order_count
    FROM orders
    GROUP BY YEAR(o_orderdate), o_orderstatus
)
ON o_orderstatus
USING SUM(order_count)
GROUP BY order_year
ORDER BY order_year
"""

duck.execute(pivot_sql).df()


### `FROM dataframe` — query a pandas DataFrame directly

DuckDB can query a Python variable that holds a pandas DataFrame as if it
were a table — no `to_sql()`, no temp table, no round-trip to disk.


In [ ]:
import pandas as pd

# Simulate a small lookup table arriving as a Python object (e.g. from an API)
priority_labels = pd.DataFrame({
    "code":  ["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"],
    "label": ["Urgent", "High", "Medium", "Standard", "Low"],
})

# Join directly against the DataFrame — no INSERT, no CREATE TABLE
duck.execute("""
    SELECT
        pl.label          AS priority,
        COUNT(*)          AS order_count,
        ROUND(SUM(o_totalprice), 2) AS total_revenue
    FROM   orders o
    JOIN   priority_labels pl ON o.o_orderpriority = pl.code
    GROUP  BY pl.label
    ORDER  BY order_count DESC
""").df()


---

## 3 · `postgres_scan` — Query Live PostgreSQL from DuckDB

DuckDB's `postgres` extension lets it attach to a running PostgreSQL instance
and query its tables directly — no export, no Parquet file, no ETL step.

This is the integration pattern used in modern data lake architectures:
DuckDB runs analytical queries over live operational data without touching
the application's primary connection pool.


In [ ]:
# Install and load the postgres extension (downloads once per machine).
duck.execute("INSTALL postgres")
duck.execute("LOAD postgres")

# Attach the PostgreSQL database as a named catalog 'pg'.
# The DSN is the same libpq string used by psycopg2 and the config module.
duck.execute(f"ATTACH '{settings.dsn}' AS pg (TYPE POSTGRES, READ_ONLY)")

print("PostgreSQL attached as catalog 'pg'")
print()

# List the tables visible through the attachment
duck.execute("SHOW ALL TABLES").df()


In [ ]:
# Query PostgreSQL tables directly from DuckDB — no data export needed.
# DuckDB pushes predicates and projections to PostgreSQL where possible.

duck.execute("""
    SELECT
        n.n_name                          AS nation,
        COUNT(DISTINCT c.c_custkey)       AS customers,
        COUNT(o.o_orderkey)               AS orders,
        ROUND(SUM(o.o_totalprice), 2)     AS total_revenue
    FROM   pg.public.orders   o
    JOIN   pg.public.customer c ON o.o_custkey   = c.c_custkey
    JOIN   pg.public.nation   n ON c.c_nationkey = n.n_nationkey
    GROUP  BY n.n_name
    ORDER  BY total_revenue DESC
""").df()


In [ ]:
# Cross-engine join: DuckDB Parquet view joined to a live PostgreSQL table.
# Demonstrates the hybrid pattern: Parquet for bulk data, PostgreSQL for
# reference / dimension tables that may be more up to date.

duck.execute(f"""
    SELECT
        pg_n.n_name                              AS nation,
        ROUND(SUM(l.l_extendedprice * (1 - l.l_discount)), 2)  AS net_revenue
    FROM   read_parquet('{PARQUET_DIR}/lineitem.parquet')  l
    JOIN   read_parquet('{PARQUET_DIR}/orders.parquet')    o
        ON l.l_orderkey = o.o_orderkey
    JOIN   pg.public.customer  pg_c ON o.o_custkey   = pg_c.c_custkey
    JOIN   pg.public.nation    pg_n ON pg_c.c_nationkey = pg_n.n_nationkey
    WHERE  l.l_shipdate BETWEEN DATE '1995-01-01' AND DATE '1995-12-31'
    GROUP  BY pg_n.n_name
    ORDER  BY net_revenue DESC
    LIMIT  10
""").df()


---

## Summary

| Feature | DuckDB | PostgreSQL |
|:---|:---|:---|
| Standard SQL (window functions, CTEs) | Yes | Yes |
| `SELECT * EXCLUDE (col)` | Yes | No |
| `SUMMARIZE` | Yes | No |
| Native `PIVOT` / `UNPIVOT` | Yes | Extension only (`crosstab`) |
| `FROM dataframe` | Yes | No |
| `postgres_scan` | Yes | N/A |
| Infrastructure required | None (`import duckdb`) | Server process |
| Best workload | Analytical, columnar, Parquet | Transactional, row-level, ACID |

**Key insight:** DuckDB does not replace PostgreSQL. It complements it.
Use PostgreSQL for writes, ACID semantics, and row-level access patterns.
Use DuckDB for analytical queries, Parquet files, and local development where
spinning up a server would add friction.

**Next:** [Notebook 06 — DuckDB + Parquet Querying](06_duckdb_parquet.ipynb)
